# Data-loader benchmarking

This notebook benchmarks the `dataloader` function and its impact on results and performance. For this purpose, we tested:
1. **Model performance:** Measured the total time elapsed when training the same model on the same data.
2. **Model accuracy:** Tested whether using `dataloader` produces any significant change in predictive power.

Results path: `/home/mriveraceron/glv-research/Results/train_benchmark`

Training data path: `/home/mriveraceron/glv-research/data_tensors/gnn_proof_full`

Model performance results are shown below.

In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import matplotlib.ticker as ticker
import os
import glob
import numpy as np

# Plotting settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'axes.linewidth': 1.2,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'legend.fontsize': 11,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

def corr_plt(data, method):
    # --- Config ---
    LIMS        = [-0.02, 1.05]
    TICK_MAJOR  = 0.2
    TICK_MINOR  = 0.05
    FMT         = '%.1f'
    # --- Data & correlations ---
    mt, mp = data['mt'], data['mp']
    r_p, _ = pearsonr(mt, mp)
    r_s, _ = spearmanr(mt, mp)
    x, y   = np.log1p(mp), np.log1p(mt)
    # --- Figure ---
    fig, ax = plt.subplots(figsize=(5, 5))
    # Hexbin
    hb = ax.hexbin(x, y, gridsize=45, cmap='YlOrRd', mincnt=1, bins='log', linewidths=0.15, edgecolors='none')
    # Colorbar
    cb = fig.colorbar(hb, ax=ax, pad=0.03, shrink=0.82, aspect=22)
    cb.set_label('Recuento (escala log)', fontsize=11, labelpad=8)
    cb.ax.tick_params(labelsize=10, direction='in')
    cb.outline.set_linewidth(0.8)
    # Identity line
    ax.plot(LIMS, LIMS, color='0.3', linewidth=1.2, linestyle='--', alpha=0.85, zorder=5, label='Correlación perfecta')
    # Labels & titles
    ax.set_xlabel('Keystoneness predicho', labelpad=8)
    ax.set_ylabel('Keystoneness esperado', labelpad=8)
    ax.set_xlim(LIMS)
    ax.set_ylim(LIMS)
    fig.suptitle('GraphConv — valores de keystoneness', fontsize=15, fontfamily='serif')
    ax.set_title(method, fontsize=11, color='0.4', pad=6)
    # Ticks (both axes)
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(ticker.MultipleLocator(TICK_MAJOR))
        axis.set_minor_locator(ticker.MultipleLocator(TICK_MINOR))
        axis.set_major_formatter(ticker.FormatStrFormatter(FMT))
    # Correlation annotation
    ax.text(
        0.05, 0.95,
        f'$r$ = {r_p.item():.3f}$\;$(Pearson)\n$\\rho$ = {r_s.item():.3f}$\;$(Spearman)',
        transform=ax.transAxes, ha='left', va='top',
        fontsize=10, fontfamily='serif', zorder=10,
        bbox=dict(boxstyle='round,pad=0.5,rounding_size=0.4',facecolor='white', edgecolor='0.75', linewidth=0.8, alpha=0.9),
    )
    # Legend & spines
    ax.legend(loc='lower right', fontsize=10, framealpha=0.9, edgecolor='0.75')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    return fig

# Script chunk for models performance
dir = '/home/mriveraceron/glv-research/Results/train_benchmark'
paths = sorted(glob.glob(f'{dir}/*.npz'))  # sorted for consistent ordering
names = ['DataLoader mezclado', 'DataLoader sin mezclar', 'orden estándar']
config = dict(zip(paths, names))

data_path, method = config
for data_path, method in config.items():
    name = os.path.basename(data_path).rsplit('.', 1)[0]
    data = np.load(data_path)
    p = corr_plt(data, method)
    save_path = f'{dir}/{name}.png'
    plt.savefig(save_path)
    plt.show()
    print(f'>> Image saved at: {save_path}')

In [ ]:
# Script chunk for model loss
import os
import glob
import numpy as np

dir = '/home/mriveraceron/glv-research/Results/train_benchmark'
paths = sorted(glob.glob(f'{dir}/*.npz'))  # sorted for consistent ordering
names = ['DataLoader mezclado', 'DataLoader sin mezclar', 'orden estándar']
colors = ['#2166ac', '#d6604d', '#4dac26']

plt.close('all')
fig, ax = plt.subplots(figsize=(8, 5))

# Colors & data
for data_path, method, color in zip(paths, names, colors):
    loss_train = np.load(data_path)['loss_history']
    epochs = range(1, len(loss_train) + 1)
    ax.plot(epochs, loss_train, color=color, linewidth=1.8, label=method, alpha=0.9)
    ax.fill_between(epochs, loss_train, alpha=0.07, color=color)

# Axes labels
ax.set_xlabel('Época', labelpad=8, fontsize=11)
ax.set_ylabel('Pérdida', labelpad=8, fontsize=11)

# Titles
fig.suptitle('Evolución de la pérdida durante el entrenamiento', fontsize=15, fontfamily='serif', y=1.01)

# Grid
ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.4)
ax.grid(True, which='minor', linestyle=':', linewidth=0.3, alpha=0.25)
ax.set_axisbelow(True)

# Ticks
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.tick_params(axis='both', labelsize=10, direction='in', length=4)

# Spines
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(0.8)

# Legend
ax.legend(loc='upper right', fontsize=10, framealpha=0.9, edgecolor='0.75', fancybox=True)

plt.tight_layout()
plt.savefig(f'{dir}/methods_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Script chunk for elapsed time
import os
import glob
import numpy as np

dir = '/home/mriveraceron/glv-research/Results/train_benchmark'
paths = sorted(glob.glob(f'{dir}/*.npz'))  # sorted for consistent ordering
names = ['DataLoader mezclado', 'DataLoader sin mezclar', 'orden estándar']


plt.clf()
plt.close('all')
fig, ax = plt.subplots(figsize=(8, 5))  # wider than tall suits time-series better

# Colors & data
colors = ['#2166ac', '#d6604d', '#4dac26']
for data_path, method, color in zip(paths, names, colors):
    elapsed_history = np.cumsum(np.load(data_path)['epochs_history'])
    epochs = range(1, len(elapsed_history) + 1)
    ax.plot(epochs, elapsed_history, color=color, linewidth=1.8,
            label=method, alpha=0.9)
    # Subtle fill under each line
    ax.fill_between(epochs, elapsed_history, alpha=0.07, color=color)

# Axes labels
ax.set_xlabel('Época', labelpad=8, fontsize=11)
ax.set_ylabel('Segundos por época', labelpad=8, fontsize=11)

# Titles
fig.suptitle('Tiempos de entrenamiento por método', fontsize=15, fontfamily='serif', y=1.01)
# Grid
ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.4)
ax.grid(True, which='minor', linestyle=':', linewidth=0.3, alpha=0.25)
ax.set_axisbelow(True)
# Ticks
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax.tick_params(axis='both', labelsize=10, direction='in', length=4)
# Spines
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(0.8)
# Legend
ax.legend(loc='upper right', fontsize=10, framealpha=0.9, edgecolor='0.75', fancybox=True)
plt.tight_layout()
plt.savefig(f'{dir}/methods_elapsed.png', dpi=150, bbox_inches='tight')
plt.show()